# Data Preprocessing

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from xgboost import XGBRegressor

# Set global font weight to bold and family to Times New Roman
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.weight'] = 'bold'
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['axes.titleweight'] = 'bold'

# Set math text to also use Times New Roman for bold formatting
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.bf'] = 'Times New Roman:bold'

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Core scikit-learn imports
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV, KFold
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# XGBoost and LightGBM
from xgboost import XGBRegressor, plot_importance
# import lightgbm as lgb

# Suppress warnings
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

## Load Dataset

In [3]:
df = pd.read_excel('All_Structures_Combined.xlsx')

print('Shape:', df.shape)
df.head()

Shape: (1800, 37)


,Structure,Window,W_Thickness,W_Eg,W_Ea,W_Dp,W_Nc,W_Nv,W_Ue,W_Uh,...,B_Nc,B_Nv,B_Ue,B_Uh,B_Na,B_dd,Voc,Jsc,FF,PCE
0,ZnS-CdTe-Sb2Se3,ZnS,0.05,3.5,4.2,10.0,1500000000000000000,1800000000000000000,50,20,...,2200000000000000000,18000000000000000000,15,5.1,10000000000000000000,100000000000000,1.133306,27.169946,89.0159,27.4097
1,ZnS-CdTe-Sb2Se3,ZnS,0.10,3.5,4.2,10.0,1500000000000000000,1800000000000000000,50,20,...,2200000000000000000,18000000000000000000,15,5.1,10000000000000000000,100000000000000,1.133291,27.154931,89.0156,27.3941
2,ZnS-CdTe-Sb2Se3,ZnS,0.15,3.5,4.2,10.0,1500000000000000000,1800000000000000000,50,20,...,2200000000000000000,18000000000000000000,15,5.1,10000000000000000000,100000000000000,1.133273,27.137083,89.0154,27.3756
3,ZnS-CdTe-Sb2Se3,ZnS,0.20,3.5,4.2,10.0,1500000000000000000,1800000000000000000,50,20,...,2200000000000000000,18000000000000000000,15,5.1,10000000000000000000,100000000000000,1.133254,27.118020,89.0153,27.3558
4,ZnS-CdTe-Sb2Se3,ZnS,0.25,3.5,4.2,10.0,1500000000000000000,1800000000000000000,50,20,...,2200000000000000000,18000000000000000000,15,5.1,10000000000000000000,100000000000000,1.133234,27.098420,89.0151,27.3355


## Encode Categorical Columns (Window, BSF)

In [4]:
# Show unique categorical values
print('Window unique values:', df['Window'].unique())
print('BSF unique values:', df['BSF'].unique())
print('Structure unique values:', df['Structure'].unique())

Window unique values: <StringArray>
['ZnS', 'CdS', 'In2S3']
Length: 3, dtype: str
BSF unique values: <StringArray>
['Sb2Se3', 'MoS2', 'ZnTe']
Length: 3, dtype: str
Structure unique values: <StringArray>
[  'ZnS-CdTe-Sb2Se3',     'ZnS-CdTe-MoS2',     'ZnS-CdTe-ZnTe',
   'CdS-CdTe-Sb2Se3',     'CdS-CdTe-MoS2',     'CdS-CdTe-ZnTe',
 'In2S3-CdTe-Sb2Se3',   'In2S3-CdTe-MoS2',   'In2S3-CdTe-ZnTe']
Length: 9, dtype: str


In [5]:
# Encode Window column
window_map = {
    'ZnS'  : 1,
    'CdS'  : 2,
    'In2S3': 3,
}
df['Window'] = df['Window'].map(window_map)

# Encode BSF column
bsf_map = {
    'Sb2Se3': 1,
    'MoS2'  : 2,
    'ZnTe'   : 3,
}
df['BSF'] = df['BSF'].map(bsf_map)

print('Window encoded:', df['Window'].unique())
print('BSF encoded   :', df['BSF'].unique())

Window encoded: [1 2 3]
BSF encoded   : [1 2 3]


## Drop Structure Column & Inspect

In [6]:
df.Structure.unique()

<StringArray>
[  'ZnS-CdTe-Sb2Se3',     'ZnS-CdTe-MoS2',     'ZnS-CdTe-ZnTe',
   'CdS-CdTe-Sb2Se3',     'CdS-CdTe-MoS2',     'CdS-CdTe-ZnTe',
 'In2S3-CdTe-Sb2Se3',   'In2S3-CdTe-MoS2',   'In2S3-CdTe-ZnTe']
Length: 9, dtype: str

In [7]:
structure_map = {
    'ZnS-CdTe-Sb2Se3': 1,
    'ZnS-CdTe-MoS2':   2,
    'ZnS-CdTe-ZnTe':    3,
    'CdS-CdTe-Sb2Se3': 4,
    'CdS-CdTe-MoS2':   5,
    'CdS-CdTe-ZnTe':    6,
    'In2S3-CdTe-Sb2Se3': 7,
    'In2S3-CdTe-MoS2':   8,
    'In2S3-CdTe-ZnTe':    9,
}

df['Structure'] = df['Structure'].map(structure_map)

In [8]:
df.Structure.unique()

array([1, 2, 3, 4, 5, 6, 7, 8, 9])

In [9]:
# # Drop the Structure name column (not an input feature)
# df.drop(columns=['Structure'], inplace=True)

# print('Columns after dropping Structure:')
# print(df.columns.tolist())

In [10]:
df.describe()

,Structure,Window,W_Thickness,W_Eg,W_Ea,W_Dp,W_Ue,W_Uh,W_dd,A_Thickness,...,B_Eg,B_Ea,B_Dp,B_Ue,B_Uh,B_dd,Voc,Jsc,FF,PCE
count,1800.000000,1800.000000,1800.000000,1800.000000,1800.000000,1800.000000,1800.000000,1800.000000,1.800000e+03,1800.00,...,1800.000000,1800.000000,1800.000000,1800.000000,1800.000000,1.800000e+03,1800.000000,1800.000000,1800.000000,1800.000000
mean,5.000000,2.000000,0.108750,2.757500,4.264583,11.983333,86.041667,24.916667,4.283300e+14,1.24,...,1.789417,3.854083,13.430500,34.341667,25.585000,4.283300e+14,1.098078,26.246351,86.368731,26.033494
std,2.582706,0.816723,0.049873,0.534145,0.097793,5.058139,30.421551,9.706290,3.600632e+15,0.00,...,0.324208,0.142198,5.843275,29.073376,39.637942,3.600632e+15,0.169956,4.039388,13.206024,4.232512
min,1.000000,1.000000,0.050000,2.000000,4.000000,5.000000,25.000000,10.000000,1.000000e+12,1.24,...,1.300000,3.400000,5.000000,5.000000,2.800000,1.000000e+12,0.000000,0.000000,0.000000,0.000000
25%,3.000000,1.000000,0.100000,2.350000,4.200000,10.000000,50.000000,20.000000,1.000000e+14,1.24,...,1.530000,3.730000,10.000000,12.000000,2.800000,1.000000e+14,1.127736,26.604575,88.606600,26.647525
50%,5.000000,2.000000,0.100000,2.400000,4.200000,10.000000,100.000000,25.000000,1.000000e+14,1.24,...,1.620000,3.800000,10.070000,15.000000,5.100000,1.000000e+14,1.132950,26.986382,88.639100,27.025300
75%,7.000000,3.000000,0.100000,3.500000,4.400000,13.500000,100.000000,25.000000,1.000000e+14,1.24,...,2.260000,4.040000,18.000000,70.000000,50.000000,1.000000e+14,1.136463,27.073530,88.972400,27.274100
max,9.000000,3.000000,0.500000,3.800000,4.450000,50.000000,250.000000,100.000000,5.000000e+16,1.24,...,2.260000,4.210000,50.000000,140.000000,280.000000,5.000000e+16,1.137285,30.548327,89.076600,28.394800


In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1800 entries, 0 to 1799
Data columns (total 37 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Structure    1800 non-null   int64  
 1   Window       1800 non-null   int64  
 2   W_Thickness  1800 non-null   float64
 3   W_Eg         1800 non-null   float64
 4   W_Ea         1800 non-null   float64
 5   W_Dp         1800 non-null   float64
 6   W_Nc         1800 non-null   object 
 7   W_Nv         1800 non-null   object 
 8   W_Ue         1800 non-null   int64  
 9   W_Uh         1800 non-null   int64  
 10  W_Na         1800 non-null   object 
 11  W_dd         1800 non-null   int64  
 12  A_Thickness  1800 non-null   float64
 13  A_Eg         1800 non-null   float64
 14  A_Ea         1800 non-null   float64
 15  A_Dp         1800 non-null   float64
 16  A_Nc         1800 non-null   int64  
 17  A_Nv         1800 non-null   int64  
 18  A_Ue         1800 non-null   int64  
 19  A_Uh         1800

## Convert All Numeric Columns to float64

In [12]:
# Convert all columns to float64 (Window and BSF are already int after mapping)
for col in df.columns:
    df[col] = df[col].astype('float64')

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1800 entries, 0 to 1799
Data columns (total 37 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Structure    1800 non-null   float64
 1   Window       1800 non-null   float64
 2   W_Thickness  1800 non-null   float64
 3   W_Eg         1800 non-null   float64
 4   W_Ea         1800 non-null   float64
 5   W_Dp         1800 non-null   float64
 6   W_Nc         1800 non-null   float64
 7   W_Nv         1800 non-null   float64
 8   W_Ue         1800 non-null   float64
 9   W_Uh         1800 non-null   float64
 10  W_Na         1800 non-null   float64
 11  W_dd         1800 non-null   float64
 12  A_Thickness  1800 non-null   float64
 13  A_Eg         1800 non-null   float64
 14  A_Ea         1800 non-null   float64
 15  A_Dp         1800 non-null   float64
 16  A_Nc         1800 non-null   float64
 17  A_Nv         1800 non-null   float64
 18  A_Ue         1800 non-null   float64
 19  A_Uh         1800

In [13]:
# Check max PCE value
print('Max PCE:', df['PCE'].max())
print('Max Voc:', df['Voc'].max())
print('Max Jsc:', df['Jsc'].max())
print('Max FF :', df['FF'].max())

Max PCE: 28.3948
Max Voc: 1.137285
Max Jsc: 30.5483268
Max FF : 89.0766


In [14]:
# df['FF'] = df['FF'] / 100

In [15]:
df.Structure.value_counts()

Structure
1.0    200
2.0    200
3.0    200
4.0    200
5.0    200
6.0    200
7.0    200
8.0    200
9.0    200
Name: count, dtype: int64

In [16]:
df.columns

Index(['Structure', 'Window', 'W_Thickness', 'W_Eg', 'W_Ea', 'W_Dp', 'W_Nc',
       'W_Nv', 'W_Ue', 'W_Uh', 'W_Na', 'W_dd', 'A_Thickness', 'A_Eg', 'A_Ea',
       'A_Dp', 'A_Nc', 'A_Nv', 'A_Ue', 'A_Uh', 'A_Na', 'A_dd', 'BSF',
       'B_Thickness', 'B_Eg', 'B_Ea', 'B_Dp', 'B_Nc', 'B_Nv', 'B_Ue', 'B_Uh',
       'B_Na', 'B_dd', 'Voc', 'Jsc', 'FF', 'PCE'],
      dtype='str')

# Train Test

In [17]:
# Features and target variables
X = df.drop(['Voc', 'Jsc', 'FF', 'PCE',
            #  "Structure"
             ]
            , axis = 1)

y = df[['Voc', 'Jsc', 'FF', 'PCE']]

# Split the dataset (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=df['Structure'], test_size=0.2, random_state = 42) 
# Equal distribution of structures in both training and testing sets

# ==================================
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (1440, 33)
X_test shape: (360, 33)
y_train shape: (1440, 4)
y_test shape: (360, 4)


In [18]:
print(X_test.Structure.value_counts())
print(X_train.Structure.value_counts())
# All Structure are equally distributed in both training and testing sets due to stratification.

Structure
9.0    40
7.0    40
5.0    40
2.0    40
3.0    40
6.0    40
4.0    40
8.0    40
1.0    40
Name: count, dtype: int64
Structure
2.0    160
4.0    160
6.0    160
9.0    160
5.0    160
3.0    160
8.0    160
7.0    160
1.0    160
Name: count, dtype: int64


In [19]:
X_train = X_train.drop(columns=["Structure"])
X_test = X_test.drop(columns=["Structure"])

# XGB Model

In [ ]:
print("="*80)
print("⚡ XGBOOST - PER-TARGET OPTIMIZED MODELS")
print("="*80)

# Best hyperparameters for each target from hyperparameter tuning
best_params_per_target_xgb = {
    'Voc': {  'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 1000},
    'Jsc': {  'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 1000},
    'FF': {   'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 1000},
    'PCE': {  'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 1000},
}

# Initialize dictionaries to store models and metrics
best_xgb_models = {}
xgb_test_predictions = {}
xgb_train_predictions = {}
metrics_dict_XGB = {}
metrics_dict_XGB_train = {}

targets = ['Voc', 'Jsc', 'FF', 'PCE']

# Convert to numpy arrays to avoid pandas compatibility issues
X_train_np = X_train.values
X_test_np = X_test.values

# Train individual XGB model for each target
for target in targets:
    print(f"\n🔨 Training XGBoost for {target}...")
    params = best_params_per_target_xgb[target]
    
    # Initialize and train XGB model with target-specific hyperparameters
    xgb_model = XGBRegressor(
        # colsample_bytree=params['colsample_bytree'],
        learning_rate=params['learning_rate'],
        max_depth=params['max_depth'],
        n_estimators=params['n_estimators'],
        # subsample=params['subsample'],
        random_state=42
    )
    
    y_train_target = y_train[[target]].values.ravel()
    y_test_target = y_test[[target]].values.ravel()
    
    xgb_model.fit(X_train_np, y_train_target)
    best_xgb_models[target] = xgb_model
    
    # --- Test Predictions ---
    xgb_test_pred = xgb_model.predict(X_test_np)
    xgb_test_mse = mean_squared_error(y_test_target, xgb_test_pred)
    xgb_test_rmse = np.sqrt(xgb_test_mse)
    xgb_test_mae = mean_absolute_error(y_test_target, xgb_test_pred)
    xgb_test_r2 = r2_score(y_test_target, xgb_test_pred)
    
    metrics_dict_XGB[target] = {
        'MSE': xgb_test_mse,
        'RMSE': xgb_test_rmse,
        'MAE': xgb_test_mae,
        'R² Score': xgb_test_r2
    }
    
    # --- Train Predictions ---
    xgb_train_pred = xgb_model.predict(X_train_np)
    xgb_train_mse = mean_squared_error(y_train_target, xgb_train_pred)
    xgb_train_rmse = np.sqrt(xgb_train_mse)
    xgb_train_mae = mean_absolute_error(y_train_target, xgb_train_pred)
    xgb_train_r2 = r2_score(y_train_target, xgb_train_pred)
    
    metrics_dict_XGB_train[target] = {
        'MSE': xgb_train_mse,
        'RMSE': xgb_train_rmse,
        'MAE': xgb_train_mae,
        'R² Score': xgb_train_r2
    }
    
    # Store predictions
    xgb_test_predictions[target] = xgb_test_pred
    xgb_train_predictions[target] = xgb_train_pred
    
    print(f"   ✅ Test R²: {xgb_test_r2:.4f} | Train R²: {xgb_train_r2:.4f}")

# Create comprehensive DataFrames
metrics_df_XGB = pd.DataFrame(metrics_dict_XGB).T
metrics_train_df_XGB = pd.DataFrame(metrics_dict_XGB_train).T

# Create full prediction DataFrames
XGB_y_pred_df = pd.DataFrame(xgb_test_predictions).T.T
XGB_y_train_pred_df = pd.DataFrame(xgb_train_predictions).T.T

# Ensure correct column order
XGB_y_pred_df.columns = targets
XGB_y_train_pred_df.columns = targets

print("\n" + "="*80)
print("📊 XGBOOST TEST SET METRICS (PER-TARGET OPTIMIZED)")
print("="*80)
print(metrics_df_XGB.to_string())

print("\n" + "="*80)
print("📊 XGBOOST TRAIN SET METRICS (PER-TARGET OPTIMIZED)")
print("="*80)
print(metrics_train_df_XGB.to_string())

print("\n✅ Per-target XGBoost models created and evaluated")

⚡ XGBOOST - PER-TARGET OPTIMIZED MODELS

🔨 Training XGBoost for Voc...
   ✅ Test R²: 0.9994 | Train R²: 0.9999

🔨 Training XGBoost for Jsc...
   ✅ Test R²: 0.9991 | Train R²: 0.9998

🔨 Training XGBoost for FF...
   ✅ Test R²: 0.9994 | Train R²: 0.9999

🔨 Training XGBoost for PCE...
   ✅ Test R²: 0.9966 | Train R²: 0.9991

📊 XGBOOST TEST SET METRICS (PER-TARGET OPTIMIZED)
          MSE      RMSE       MAE  R² Score
Voc  0.000016  0.003972  0.001672  0.999392
Jsc  0.012516  0.111877  0.037170  0.999107
FF   0.094349  0.307163  0.113476  0.999382
PCE  0.051823  0.227646  0.100214  0.996620

📊 XGBOOST TRAIN SET METRICS (PER-TARGET OPTIMIZED)
          MSE      RMSE       MAE  R² Score
Voc  0.000004  0.001994  0.001028  0.999866
Jsc  0.003995  0.063205  0.021986  0.999763
FF   0.010162  0.100807  0.049733  0.999943
PCE  0.016708  0.129259  0.059151  0.999099

✅ Per-target XGBoost models created and evaluated


# Finding Best Optimized Window and BSF Material 

In [21]:
import pandas as pd

# ---------- Structure list ----------
STRUCTURES = {
    1: "ZnS-CdTe-Sb2Se3",
    2: "ZnS-CdTe-MoS2",
    3: "ZnS-CdTe-ZnTe",
    4: "CdS-CdTe-Sb2Se3",
    5: "CdS-CdTe-MoS2",
    6: "CdS-CdTe-ZnTe",
    7: "In2S3-CdTe-Sb2Se3",
    8: "In2S3-CdTe-MoS2",
    9: "In2S3-CdTe-ZnTe",
}

ABSORBER_PARAMS = {
    "A_Thickness": 1.24, "A_Eg": 1.53, "A_Ea": 4.28, "A_Dp": 10.3,
    "A_Nc": 9.20e+17, "A_Nv": 5.20e+18, "A_Ue": 320, "A_Uh": 40,
    "A_Na": 1.00e+16, "A_dd": 1.00e+14,
}

WINDOW_BASELINES = {
    "ZnS": {
        "W_Thickness": 0.1, "W_Eg": 3.5, "W_Ea": 4.2, "W_Dp": 10,
        "W_Nc": 1.50e+18, "W_Nv": 1.80e+18, "W_Ue": 50, "W_Uh": 20,
        "W_Na": 1.00e+18, "W_dd": 1.00e+14,
    },
    "In2S3": {
        "W_Thickness": 0.1, "W_Eg": 2.35, "W_Ea": 4.4, "W_Dp": 13.5,
        "W_Nc": 2.20e+17, "W_Nv": 1.80e+19, "W_Ue": 100, "W_Uh": 25,
        "W_Na": 1.00e+18, "W_dd": 1.00e+14,
    },
    "CdS": {
        "W_Thickness": 0.1, "W_Eg": 2.4, "W_Ea": 4.2, "W_Dp": 10,
        "W_Nc": 2.20e+18, "W_Nv": 1.80e+19, "W_Ue": 100, "W_Uh": 25,
        "W_Na": 1.00e+18, "W_dd": 1.00e+14,
    },
}

BSF_BASELINES = {
    "Sb2Se3": {
        "B_Thickness": 0.2, "B_Eg": 1.53, "B_Ea": 4.04, "B_Dp": 18,
        "B_Nc": 2.20e+18, "B_Nv": 1.80e+19, "B_Ue": 15, "B_Uh": 5.1,
        "B_Na": 1.00e+19, "B_dd": 1.00e+14,
    },
    "MoS2": {
        "B_Thickness": 0.2, "B_Eg": 1.62, "B_Ea": 3.8, "B_Dp": 10.07,
        "B_Nc": 2.80e+19, "B_Nv": 1.00e+19, "B_Ue": 12, "B_Uh": 2.8,
        "B_Na": 1.00e+19, "B_dd": 1.00e+14,
    },
    "ZnTe": {
        "B_Thickness": 0.2, "B_Eg": 2.26, "B_Ea": 3.73, "B_Dp": 10.0,
        "B_Nc": 7.5e+16, "B_Nv": 1.5e+19, "B_Ue": 70, "B_Uh": 50,
        "B_Na": 1.00e+19, "B_dd": 1.00e+14,
    },
}

# ---------- Numeric codes for Window / BSF ----------
# EDIT these to match the exact codes you used when training the model
WINDOW_MAP = {"ZnS": 1, "CdS": 2, "In2S3": 3}
BSF_MAP    = {"Sb2Se3": 1, "MoS2": 2, "ZnTe": 3}

# ---------- Column order (same as training) ----------
COLUMNS = [
    'Structure', 'Window',
    'W_Thickness', 'W_Eg', 'W_Ea', 'W_Dp', 'W_Nc', 'W_Nv', 'W_Ue', 'W_Uh', 'W_Na', 'W_dd',
    'A_Thickness', 'A_Eg', 'A_Ea', 'A_Dp', 'A_Nc', 'A_Nv', 'A_Ue', 'A_Uh', 'A_Na', 'A_dd',
    'BSF',
    'B_Thickness', 'B_Eg', 'B_Ea', 'B_Dp', 'B_Nc', 'B_Nv', 'B_Ue', 'B_Uh', 'B_Na', 'B_dd',
]

# ---------- Build the 9-row DataFrame ----------
rows = []
for sid, name in STRUCTURES.items():
    window, _, bsf = name.split("-")
    row = {"Structure": sid, "Window": WINDOW_MAP[window], "BSF": BSF_MAP[bsf]}
    row.update(WINDOW_BASELINES[window])
    row.update(ABSORBER_PARAMS)
    row.update(BSF_BASELINES[bsf])
    rows.append(row)

df_structures = pd.DataFrame(rows)[COLUMNS]
df_structures.insert(1, "Structure_Name", [STRUCTURES[i] for i in df_structures["Structure"]])

print(df_structures.to_string())

   Structure     Structure_Name  Window  W_Thickness  W_Eg  W_Ea  W_Dp          W_Nc          W_Nv  W_Ue  W_Uh          W_Na          W_dd  A_Thickness  A_Eg  A_Ea  A_Dp          A_Nc          A_Nv  A_Ue  A_Uh          A_Na          A_dd  BSF  B_Thickness  B_Eg  B_Ea   B_Dp          B_Nc          B_Nv  B_Ue  B_Uh          B_Na          B_dd
0          1    ZnS-CdTe-Sb2Se3       1          0.1  3.50   4.2  10.0  1.500000e+18  1.800000e+18    50    20  1.000000e+18  1.000000e+14         1.24  1.53  4.28  10.3  9.200000e+17  5.200000e+18   320    40  1.000000e+16  1.000000e+14    1          0.2  1.53  4.04  18.00  2.200000e+18  1.800000e+19    15   5.1  1.000000e+19  1.000000e+14
1          2      ZnS-CdTe-MoS2       1          0.1  3.50   4.2  10.0  1.500000e+18  1.800000e+18    50    20  1.000000e+18  1.000000e+14         1.24  1.53  4.28  10.3  9.200000e+17  5.200000e+18   320    40  1.000000e+16  1.000000e+14    2          0.2  1.62  3.80  10.07  2.800000e+19  1.000000e+19    12   2.8

In [22]:
import numpy as np
import pandas as pd

targets = ['Voc', 'Jsc', 'FF', 'PCE']

# 1) Select the 32 model features in the exact training order
feature_cols = list(X_test.columns)
X_struct = df_structures[feature_cols].values   # numpy array, as the models were trained on numpy

# 2) Predict all 4 targets for the 9 structures
results = df_structures[["Structure", "Structure_Name"]].copy()
for target in targets:
    results[f"{target}_pred"] = best_xgb_models[target].predict(X_struct)       # Predict using the trained XGB model for each target

# 3) Rank by predicted PCE
results = results.sort_values("PCE_pred", ascending=False).reset_index(drop=True)
results.insert(0, "Rank", results.index + 1)

print("=" * 80)
print(" PREDICTED PERFORMANCE OF THE 9 STRUCTURES (ranked by PCE)")
print("=" * 80)
print(results.round(4).to_string(index=False))

# 4) Best structure
best = results.iloc[0]
print("\n" + "=" * 80)
print(f" BEST STRUCTURE: {best['Structure_Name']} (Structure #{int(best['Structure'])})")
print(f"   Voc = {best['Voc_pred']:.4f} V | Jsc = {best['Jsc_pred']:.4f} mA/cm² "
      f"| FF = {best['FF_pred']:.4f} | PCE = {best['PCE_pred']:.4f} %")
print("=" * 80)

# 5) Optional: save the table
# results.to_csv("structure_predictions.csv", index=False)

 PREDICTED PERFORMANCE OF THE 9 STRUCTURES (ranked by PCE)
 Rank  Structure    Structure_Name  Voc_pred  Jsc_pred   FF_pred  PCE_pred
    1          1   ZnS-CdTe-Sb2Se3    1.1326 27.151100 89.005600 27.400000
    2          7 In2S3-CdTe-Sb2Se3    1.1331 27.053600 89.006599 27.297199
    3          3     ZnS-CdTe-ZnTe    1.1364 27.073700 88.654503 27.270300
    4          9   In2S3-CdTe-ZnTe    1.1371 26.969101 88.635902 27.156700
    5          2     ZnS-CdTe-MoS2    1.1288 27.092501 88.642502 27.100700
    6          8   In2S3-CdTe-MoS2    1.1283 26.987900 88.628998 26.985901
    7          6     CdS-CdTe-ZnTe    1.1369 26.581499 88.612198 26.772200
    8          4   CdS-CdTe-Sb2Se3    1.1281 26.628201 88.999001 26.740499
    9          5     CdS-CdTe-MoS2    1.1278 26.601601 88.646301 26.602600

 BEST STRUCTURE: ZnS-CdTe-Sb2Se3 (Structure #1)
   Voc = 1.1326 V | Jsc = 27.1511 mA/cm² | FF = 89.0056 | PCE = 27.4000 %


In [35]:
import itertools
import pandas as pd

# =====================================================================
# PARAMETER SPACE  (list = varying | single number = constant)
# =====================================================================
# Finding best combination of Window (ZnS) and BSF (Sb2Se3) parameters particularly Eg and Ea for maximum PCE
param_space = {
    # ---------------- Window (ZnS constants, Eg & Ea varying) ----------------
    "Window":      1,                       # <-- code of ZnS (set to YOUR training code)
    "W_Thickness": 0.1,
    "W_Eg":        [2.3, 2.6, 2.9, 3.2, 3.5],        # VARYING
    "W_Ea":        [4.1, 4.2, 4.3, 4.4],              # VARYING
    "W_Dp":        10,
    "W_Nc":        1.50e+18,
    "W_Nv":        1.80e+18,
    "W_Ue":        50,
    "W_Uh":        20,
    "W_Na":        1.00e+18,
    "W_dd":        1.00e+14,

    # ---------------- Absorber (CdTe, all constant) ----------------
    "A_Thickness": 1.24,
    "A_Eg":        1.53,
    "A_Ea":        4.28,
    "A_Dp":        10.3,
    "A_Nc":        9.20e+17,
    "A_Nv":        5.20e+18,
    "A_Ue":        320,
    "A_Uh":        40,
    "A_Na":        1.00e+16,
    "A_dd":        1.00e+14,

    # ---------------- BSF (Sb2Se3 constants, Eg & Ea varying) ----------------
    "BSF":         1,                       # <-- code of Sb2Se3 (set to YOUR training code)
    "B_Thickness": 0.2,
    "B_Eg":        [ 1.4, 1.45, 1.5],      # VARYING
    "B_Ea":        [3.6, 3.7, 3.8, 3.9, 4.04],       # VARYING
    "B_Dp":        18,
    "B_Nc":        2.20e+18,
    "B_Nv":        1.80e+19,
    "B_Ue":        15,
    "B_Uh":        5.1,
    "B_Na":        1.00e+19,
    "B_dd":        1.00e+14,
}

# =====================================================================
# BUILD GRID
# =====================================================================
feature_cols = list(X_test.columns)

# safety check: every model feature must be defined above
missing = set(feature_cols) - set(param_space)
extra   = set(param_space) - set(feature_cols)
assert not missing, f"Missing features: {missing}"
assert not extra,   f"Extra features not in model: {extra}"

# wrap constants into 1-element lists, keep training column order
value_lists = [
    v if isinstance(v, (list, tuple)) else [v]
    for v in (param_space[c] for c in feature_cols)
]

df_grid = pd.DataFrame(list(itertools.product(*value_lists)), columns=feature_cols)

varying_cols = [c for c in feature_cols if isinstance(param_space[c], (list, tuple))]
print(f"Total combinations: {len(df_grid)}")
print(f"Varying features : {varying_cols}")

# =====================================================================
# PREDICT
# =====================================================================
targets = ['Voc', 'Jsc', 'FF', 'PCE']
X_grid = df_grid[feature_cols].values

for t in targets:
    df_grid[f"{t}_pred"] = best_xgb_models[t].predict(X_grid)

# =====================================================================
# TOP 5 BY PCE
# =====================================================================
top5 = df_grid.sort_values("PCE_pred", ascending=False).head(5).reset_index(drop=True)
print(top5[varying_cols + [f"{t}_pred" for t in targets]].round(4).to_string(index=False))

Total combinations: 300
Varying features : ['W_Eg', 'W_Ea', 'B_Eg', 'B_Ea']
 W_Eg  W_Ea  B_Eg  B_Ea  Voc_pred  Jsc_pred   FF_pred  PCE_pred
  3.5   4.2  1.50  4.04    1.1365 27.271999 89.021797 27.527300
  3.5   4.1  1.50  4.04    1.1365 27.251400 89.021004 27.524099
  3.5   4.2  1.40  4.04    1.0920 28.517099 88.590103 27.447800
  3.5   4.1  1.40  4.04    1.0920 28.496500 88.589203 27.444599
  3.5   4.2  1.45  4.04    1.1137 27.764299 88.849503 27.289301
